# Data Migration SQL Server to Postgres

In [1]:
import os
import pandas as pd
import pyodbc
import psycopg2
from psycopg2.extras import execute_values
from dotenv import load_dotenv

## 1. Load Credential environment variables

In [2]:
load_dotenv()  # Load environment variables from .env file

True

In [3]:
sql_host = os.getenv("SQL_SERVER_HOST")
sql_database = os.getenv("SQL_SERVER_DB")
sql_user = os.getenv("SQL_SERVER_USER")

In [4]:
print(f"SQL Server Host: {sql_host}")
print(f"SQL Server Database: {sql_database}")
print(f"SQL Server User: {sql_user}")

SQL Server Host: (local)\SQLEXPRESS
SQL Server Database: transactionDB_UAT
SQL Server User: windows


In [5]:
pg_host=os.getenv("PROGRES_HOST")
pg_port=os.getenv("PROGRES_PORT")
pg_database=os.getenv("PROGRES_DB")
pg_user=os.getenv("PROGRES_USER")
pg_password=os.getenv("PROGRES_PASSWORD") 

In [6]:
print (f"PostgreSQL Host: {pg_host}")
print (f"PostgreSQL Port: {pg_port}")
print (f"PostgreSQL Database: {pg_database}")
print (f"PostgreSQL User: {pg_user}")
print (f"PostgreSQL Password: {pg_password}")

PostgreSQL Host: localhost
PostgreSQL Port: 5432
PostgreSQL Database: transaction_uat
PostgreSQL User: postgres
PostgreSQL Password: 123456


## 2. Connect to SQL Server

In [8]:
print("Connecting to SQL Server...")
print(f"    Server: {sql_host}")
print(f"    Database: {sql_database}")

Connecting to SQL Server...
    Server: (local)\SQLEXPRESS
    Database: transactionDB_UAT


In [9]:
try:
    sql_conn_string = (
        f"DRIVER={{ODBC Driver 17 for SQL Server}};"
        f"SERVER={sql_host};"
        f"DATABASE={sql_database};"
        f"Trusted_Connection=yes;"
    )

    sql_conn = pyodbc.connect(sql_conn_string)
    sql_cursor = sql_conn.cursor()
    print("[SUCCESS] Connected to SQL Server")
except Exception as e:
    print(f"Error connecting to SQL Server: {e}")
    print(""" How to troubleshoot:
        >1. Check server name in .env file is correct
        >2. Verify that the SQL Server is running
        >3. Ensure that the ODBC Driver 17 for SQL Server is installed
        >4. Check if you can connect to the SQL Server using SQL Server Management Studio (SSMS) or another database client
        >5. If using a remote SQL Server, ensure that the server allows remote connections and that your IP address is whitelisted
        >6. If using Windows Authentication, ensure that your Windows user has the necessary permissions to access the SQL Server database
        >7. If using SQL Server Authentication, ensure that the username and password in the .env file are correct
        >8. Check if there are any firewall rules blocking the connection to the SQL Server
        >9. If the SQL Server is running on a non-default port, make sure to specify the correct port in the .env file
          """)

[SUCCESS] Connected to SQL Server


## 3. Connect to PostgresSQL

In [10]:
print("Connecting to PostgreSQL...")
print(f"    Server: {pg_host}")
print(f"    Database: {pg_database}")

Connecting to PostgreSQL...
    Server: localhost
    Database: transaction_uat


In [14]:
try:
    pg_conn = psycopg2.connect(
        host = pg_host,
        port = pg_port,
        database = pg_database,
        user = pg_user,
        password = pg_password
)
    pg_cursor = pg_conn.cursor()
    pg_cursor.execute("SELECT version();")
    pg_version = pg_cursor.fetchone()[0]

    print(f"[SUCCESS] Connected to PostgreSQL: {pg_version[:50]}...\n")

except psycopg2.OperationalError as e:
    print(f"[ERROR] Failed to connect to PostgreSQL: {e}")
    print(""" How to troubleshoot:
        >1. Check if PostgreSQL server is running and accessible
        >2. Check server name, port, database name, username, and password in
        >3. Check database existence and user permissions
        >4. Check for firewall or network issues
        >5. Ensure psycopg2 library is installed and up to date
        >6. If using a cloud-hosted PostgreSQL, ensure that your IP address is whitelisted
        >7. If using SSL, ensure that the SSL settings in the .env file are correct
        >8. If using a connection string, ensure that it is formatted correctly and contains all necessary parameters
        >9. If using a connection pool, ensure that the pool is configured correctly and has available connections
          """)
except Exception as e:
    print(f"[ERROR] An unexpected error occurred while connecting to PostgreSQL: {e}")
    raise

[SUCCESS] Connected to PostgreSQL: PostgreSQL 18.4 on x86_64-windows, compiled by msv...



# 4. Define the tables to migrate

## Migration order

 - Categories (no dependencies)
 - Suppliers (no dependencies)
 - Customers (no dependencies)
 - Products (depends on Categories and Suppliers)

In [ ]:
tables_to_migrate = ['Categories', 'Suppliers', 'Customers', 'Products']

Tables to migrate: ['Categories', 'Suppliers', 'Customers', 'Products']


In [20]:
print("Tables to migrate:")
for i, table in enumerate(tables_to_migrate,1):
    print(f" {i}. {table}")

total_no_tables = len(tables_to_migrate)
print(f"\nTotal no of tables to migrate: {total_no_tables}")

Tables to migrate:
 1. Categories
 2. Suppliers
 3. Customers
 4. Products

Total no of tables to migrate: 4
